# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Imhs14/flyrank-ml-internship-starter-main/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a clustering problem, because you're grouping pages by shared structural traits without any pre-existing 'correct' label to learn from. we will let the computer do the grouping by itself.we then judge how well-separated and meaningful those groups turn out to be.

In [24]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    !git clone https://github.com/Imhs14/flyrank-ml-internship-starter-main.git repo
    %cd repo

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

# Check: does a "correct" label already exist for me to predict?
possible_label_cols = [c for c in df.columns if "cluster" in c.lower() or "archetype" in c.lower() or "segment" in c.lower()]
print("Pre-existing label/cluster columns found:", possible_label_cols)
print(f"None of the {len(df.columns)} columns are a ready-made archetype — confirms this is unsupervised.")



(30000, 44)
Pre-existing label/cluster columns found: []
None of the 44 columns are a ready-made archetype — confirms this is unsupervised.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There's no traditional target column. instead, the 'proxy' is the cluster/archetype assignment your model will produce, based on structural features for each page(check docs/data dictionary.md).e.g. word count, internal links, media count, page type.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df.columns.tolist()
proxy_feature_candidates = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
df[proxy_feature_candidates].describe()

,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
count,22301.000000,22301.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000
mean,3107.760325,20665.277835,256.16780,46.098300,5200.366300,16.097333,0.510733,16.34238,2.534520,18.212921,0.768196
std,1452.382598,10115.344042,132.70793,42.078709,16838.019547,75.076958,3.279162,15.21679,8.310096,29.472768,7.429454
min,8.000000,40.000000,90.00000,1.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
25%,2413.000000,15644.000000,132.00000,20.000000,81.000000,0.000000,0.000000,6.20000,0.000000,0.000000,0.000000
50%,2877.000000,19116.000000,236.00000,20.000000,731.000000,1.000000,0.070000,10.80000,0.000000,5.000000,0.000000
75%,3666.000000,24011.000000,333.00000,104.000000,3615.250000,7.000000,0.290000,22.30000,1.350000,23.530000,0.000000
max,9546.000000,111158.000000,564.00000,373.000000,517715.000000,4178.000000,100.000000,245.00000,100.000000,300.000000,300.000000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Since there's no correct answer to check clusters against, I'll use silhouette score as my success metric. It's a number between -1 and 1 that measures, for each page, how much closer it is to pages in its own cluster than to pages in the nearest other cluster. A score closer to 1 means the archetypes are tight and clearly separated from each other; a score near 0 means the groups overlap and aren't really distinct; negative means pages are probably in the wrong group. I'll treat a silhouette score meaningfully above 0 — and ideally above 0.3-0.5 — as a sign the archetypes are real groupings and not just noise."

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check: does a "correct" label already exist for me to predict?
possible_label_cols = [c for c in df.columns if "cluster" in c.lower() or "archetype" in c.lower() or "segment" in c.lower()]
print("Pre-existing label/cluster columns found:", possible_label_cols)
print(f"None of the {len(df.columns)} columns are a ready-made archetype — confirms this is unsupervised.")

proxy_feature_candidates = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
df[proxy_feature_candidates].describe()

Pre-existing label/cluster columns found: []
None of the 44 columns are a ready-made archetype — confirms this is unsupervised.


,word_count,char_count,content_age_days,days_since_last_update,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct
count,22301.000000,22301.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000
mean,3107.760325,20665.277835,256.16780,46.098300,5200.366300,16.097333,0.510733,16.34238,2.534520,18.212921,0.768196
std,1452.382598,10115.344042,132.70793,42.078709,16838.019547,75.076958,3.279162,15.21679,8.310096,29.472768,7.429454
min,8.000000,40.000000,90.00000,1.000000,1.000000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000
25%,2413.000000,15644.000000,132.00000,20.000000,81.000000,0.000000,0.000000,6.20000,0.000000,0.000000,0.000000
50%,2877.000000,19116.000000,236.00000,20.000000,731.000000,1.000000,0.070000,10.80000,0.000000,5.000000,0.000000
75%,3666.000000,24011.000000,333.00000,104.000000,3615.250000,7.000000,0.290000,22.30000,1.350000,23.530000,0.000000
max,9546.000000,111158.000000,564.00000,373.000000,517715.000000,4178.000000,100.000000,245.00000,100.000000,300.000000,300.000000


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
proxy_feature_candidates = [
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
df[proxy_feature_candidates].describe()
from sklearn.metrics import silhouette_score
print(silhouette_score.__doc__.split("\n\n")[0])  # confirms what it needs: a feature matrix + cluster labels


Compute the mean Silhouette Coefficient of all samples.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A "fixed rule" would be somthing like: if word_count< 300, call it thin content. That's brittle and only looks at one thing at a time. clustering can consider many traits together and find groupings a human wouldn't think to hand-write. 

In [34]:
lane_cols = [
    "content_id", "content_type", "word_count", "char_count", "content_age_days",
    "days_since_last_update", "impressions_90d", "clicks_90d", "ctr",
    "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
df_lane = df[lane_cols]
print(f"One row = one content item (page). Shape: {df_lane.shape}")
df_lane.head()

df.groupby("word_count_tier")["engagement_rate"].agg(["mean", "std", "count"])

One row = one content item (page). Shape: (30000, 13)


,mean,std,count
word_count_tier,,,
1000-2000,2.666574,9.570179,3780
2000-3500,3.126750,8.913164,11263
3500+,1.464084,4.161736,6285
<1000,2.960740,14.672483,973


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.